In [1]:
!pip install selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 33.3 MB/s  0:00:0034.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [selenium]━━ 5/6 [selenium]


In [24]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from bs4 import BeautifulSoup

import pandas as pd
import re
import os

from datetime import datetime

In [25]:


url = "https://www.purdue.edu/recwell/facility-usage/index.php"

driver = webdriver.Chrome()

driver.get(url)

#Give the page time to load the facility data
time.sleep(5)

print(driver.page_source[:2000])

<html xmlns="http://www.w3.org/1999/xhtml" lang="en-US"><head>
<title>
Facility Usage | Purdue University
</title>
<link href="https://www.purdue.edu/recwell/facility-usage/index.php" rel="canonical">

<meta charset="utf-8">
<meta content="IE=edge" http-equiv="X-UA-Compatible">
<meta content="width=device-width,initial-scale=1" name="viewport">

 <!-- Google tag (gtag.js) --> <script type="text/javascript" async="" src="https://www.googletagmanager.com/gtag/destination?id=DC-8141015&amp;cx=c&amp;gtm=4e6a61"></script><script type="text/javascript" async="" src="//siteimproveanalytics.com/js/siteanalyze_6044943.js"></script><script src="https://connect.facebook.net/signals/config/1290758077759811?v=2.9.415&amp;r=stable&amp;domain=www.purdue.edu&amp;hme=a4c28c8787325ac3f8f9eac68364f47df2a8c4266e43177443e5d7f6289c9774&amp;ex_m=117%2C240%2C170%2C27%2C81%2C82%2C161%2C77%2C76%2C11%2C180%2C101%2C20%2C152%2C140%2C46%2C84%2C89%2C148%2C175%2C182%2C8%2C4%2C5%2C7%2C6%2C3%2C102%2C114%2C32%2C17%2C31%

In [3]:
print(driver.find_element(By.TAG_NAME, "body").text)

Find Info For
Apply
News
President
Shop
Visit
Give
Emergency
Recreation & Wellness
MENU
Are you trying to find information for Purdue in Indianapolis? Visit our Purdue in Indianapolis webpage for more information.
Facility Usage
Facility Usage
Faculty/Staff Workout Area
Capacity: 4/25 // 16%
Last Updated: Oct-07 04:53pm
Upper Track (4th Floor)
Capacity: 1/10 // 10%
Last Updated: Oct-07 04:53pm
Upper Gym
Capacity: 6/100 // 6%
Last Updated: Oct-07 04:53pm
MP6
Capacity: 5/45 // 12%
Last Updated: Oct-07 04:53pm
MP7
Capacity: 0/45 // 0%
Last Updated: Oct-07 04:53pm
Upper Fitness
Capacity: 12/46 // 27%
Last Updated: Oct-07 04:53pm
MAC Court
Capacity: 12/100 // 12%
Last Updated: Oct-07 04:53pm
East Mezzanine (3rd Floor)
Capacity: 6/15 // 40%
Last Updated: Oct-07 04:53pm
Atrium Track (3rd Floor)
Capacity: 0/50 // 0%
Last Updated: Oct-07 04:53pm
West Mezzanine (3rd Floor)
Capacity: 4/25 // 16%
Last Updated: Oct-07 04:53pm
Mindfulness Room
Capacity: 1/20 // 5%
Last Updated: Oct-07 04:53pm
Racque

In [6]:
#extracting individual facilities 
body_text = driver.find_element(By.TAG_NAME, "body").text

print(body_text)

lines = body_text.splitlines()

for line in lines:
    if "Capacity:" in line:
        print(line)

Find Info For
Apply
News
President
Shop
Visit
Give
Emergency
Recreation & Wellness
MENU
Are you trying to find information for Purdue in Indianapolis? Visit our Purdue in Indianapolis webpage for more information.
Facility Usage
Facility Usage
Faculty/Staff Workout Area
Capacity: 4/25 // 16%
Last Updated: Oct-07 04:53pm
Upper Track (4th Floor)
Capacity: 1/10 // 10%
Last Updated: Oct-07 04:53pm
Upper Gym
Capacity: 6/100 // 6%
Last Updated: Oct-07 04:53pm
MP6
Capacity: 5/45 // 12%
Last Updated: Oct-07 04:53pm
MP7
Capacity: 0/45 // 0%
Last Updated: Oct-07 04:53pm
Upper Fitness
Capacity: 12/46 // 27%
Last Updated: Oct-07 04:53pm
MAC Court
Capacity: 12/100 // 12%
Last Updated: Oct-07 04:53pm
East Mezzanine (3rd Floor)
Capacity: 6/15 // 40%
Last Updated: Oct-07 04:53pm
Atrium Track (3rd Floor)
Capacity: 0/50 // 0%
Last Updated: Oct-07 04:53pm
West Mezzanine (3rd Floor)
Capacity: 4/25 // 16%
Last Updated: Oct-07 04:53pm
Mindfulness Room
Capacity: 1/20 // 5%
Last Updated: Oct-07 04:53pm
Racque

In [7]:
#parse those numbers: 
import re

for line in lines:
    if "Capacity:" in line:

        match = re.search(
            r"Capacity:\s*(\d+)/(\d+)\s*//\s*(\d+)%",
            line
        )

        if match:

            current = int(match.group(1))
            capacity = int(match.group(2))
            percent = int(match.group(3))

            print(current, capacity, percent)

4 25 16
1 10 10
6 100 6
5 45 12
0 45 0
12 46 27
12 100 12
6 15 40
0 50 0
4 25 16
1 20 5
0 80 0
18 106 17
23 70 33
10 55 19
3 50 6
15 100 15
11 100 11
29 80 37
0 100 0
7 26 27
4 20 20
26 208 13
2 16 13
5 20 25
0 15 0
3 11 28
4 20 20
1 26 4
53 380 14
68 282 25
0 50 0
26 50 52


In [8]:
#inspect the rendered HTML structure:
from bs4 import BeautifulSoup

html = driver.page_source
soup = BeautifulSoup(html, "html.parser")

print(soup.prettify()[:10000])

<html lang="en-US" xmlns="http://www.w3.org/1999/xhtml">
 <head>
  <title>
   Facility Usage | Purdue University
  </title>
  <link href="https://www.purdue.edu/recwell/facility-usage/index.php" rel="canonical"/>
  <meta charset="utf-8"/>
  <meta content="IE=edge" http-equiv="X-UA-Compatible"/>
  <meta content="width=device-width,initial-scale=1" name="viewport"/>
  <!-- Google tag (gtag.js) -->
  <script async="" src="https://www.googletagmanager.com/gtag/destination?id=DC-8141015&amp;cx=c&amp;gtm=4e6a60" type="text/javascript">
  </script>
  <script async="" src="//siteimproveanalytics.com/js/siteanalyze_6044943.js" type="text/javascript">
  </script>
  <script async="" src="https://connect.facebook.net/signals/config/1290758077759811?v=2.9.415&amp;r=stable&amp;domain=www.purdue.edu&amp;hme=a4c28c8787325ac3f8f9eac68364f47df2a8c4266e43177443e5d7f6289c9774&amp;ex_m=117%2C240%2C170%2C27%2C81%2C82%2C161%2C77%2C76%2C11%2C180%2C101%2C20%2C152%2C140%2C46%2C84%2C89%2C148%2C175%2C182%2C8%2C4%

In [9]:
text = soup.find(string=lambda x: x and "Upper Fitness" in x)

print(text)
print(text.parent)

Upper Fitness 
<h5 class="rw-c2c-feed__location--name">Upper Fitness </h5>


In [10]:
print(text.parent.parent)

<div class="rw-c2c-feed__location"><h5 class="rw-c2c-feed__location--name">Upper Fitness </h5><div aria-valuemax="100" aria-valuemin="0" aria-valuenow="27" class="MuiLinearProgress-root MuiLinearProgress-colorPrimary rw-c2c-feed__location--progress MuiLinearProgress-determinate" role="progressbar"><div class="MuiLinearProgress-bar MuiLinearProgress-barColorPrimary MuiLinearProgress-bar1Determinate" style="transform: translateX(-73%);"></div></div><div class="rw-c2c-feed__location--about"><span class="rw-c2c-feed__about--capacity">Capacity: 12/46 // 27%</span><span class="rw-c2c-feed__about--update">Last Updated: Oct-07 04:53pm</span></div></div>


In [11]:
#finding number of facilities -- checking if it works 
from selenium import webdriver
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
from datetime import datetime

url = "https://www.purdue.edu/recwell/facility-usage/index.php"

#Open Chrome
driver = webdriver.Chrome()

#Open Purdue facility usage page
driver.get(url)

#Wait for JavaScript to load the facility data
time.sleep(5)

#Get the fully rendered HTML
html = driver.page_source

#Parse the HTML
soup = BeautifulSoup(html, "html.parser")

#Find every facility
facility_cards = soup.find_all("div", class_="rw-c2c-feed__location")

print("Facilities found:", len(facility_cards))

Facilities found: 37


In [13]:
#scraping each facility:
facilities = []

for card in facility_cards:

    #Find the pieces inside each facility card
    name_tag = card.find(
        "h5",
        class_="rw-c2c-feed__location--name"
    )

    capacity_tag = card.find(
        "span",
        class_="rw-c2c-feed__about--capacity"
    )

    updated_tag = card.find(
        "span",
        class_="rw-c2c-feed__about--update"
    )

    #Skip cards that don't have the information we need
    if name_tag is None or capacity_tag is None:
        continue

    #Extract the text
    name = name_tag.get_text(strip=True)
    capacity_text = capacity_tag.get_text(strip=True)

    #Some cards may not have an update time
    if updated_tag is not None:
        updated = updated_tag.get_text(strip=True)
    else:
        updated = ""

    #Extract current people, capacity, and percentage
    match = re.search(
        r"Capacity:\s*(\d+)/(\d+)\s*//\s*(\d+)%",
        capacity_text
    )

    if match:

        current = int(match.group(1))
        capacity = int(match.group(2))
        percent = int(match.group(3))

        facilities.append({
            "timestamp": datetime.now(),
            "facility": name,
            "current": current,
            "capacity": capacity,
            "percent": percent,
            "last_updated": updated.replace("Last Updated:", "").strip()
        })

In [16]:
df = pd.DataFrame(facilities)

df

,timestamp,facility,current,capacity,percent,last_updated
0,2026-10-07 17:38:28.048160,Faculty/Staff Workout Area,5,25,20,Oct-07 05:25pm
1,2026-10-07 17:38:28.048266,Upper Track (4th Floor),1,10,10,Oct-07 05:25pm
2,2026-10-07 17:38:28.048340,Upper Gym,11,100,11,Oct-07 05:25pm
3,2026-10-07 17:38:28.048412,MP6,0,45,0,Oct-07 05:25pm
4,2026-10-07 17:38:28.048486,MP7,0,45,0,Oct-07 05:25pm
5,2026-10-07 17:38:28.048559,Upper Fitness,23,46,50,Oct-07 05:25pm
6,2026-10-07 17:38:28.048629,MAC Court,18,100,18,Oct-07 05:25pm
7,2026-10-07 17:38:28.048691,East Mezzanine (3rd Floor),5,15,34,Oct-07 05:25pm
8,2026-10-07 17:38:28.048756,Atrium Track (3rd Floor),2,50,4,Oct-07 05:25pm
9,2026-10-07 17:38:28.048820,West Mezzanine (3rd Floor),7,25,29,Oct-07 05:25pm


In [16]:
df = pd.DataFrame(facilities)

df

,timestamp,facility,current,capacity,percent,last_updated
0,2026-10-07 17:38:28.048160,Faculty/Staff Workout Area,5,25,20,Oct-07 05:25pm
1,2026-10-07 17:38:28.048266,Upper Track (4th Floor),1,10,10,Oct-07 05:25pm
2,2026-10-07 17:38:28.048340,Upper Gym,11,100,11,Oct-07 05:25pm
3,2026-10-07 17:38:28.048412,MP6,0,45,0,Oct-07 05:25pm
4,2026-10-07 17:38:28.048486,MP7,0,45,0,Oct-07 05:25pm
5,2026-10-07 17:38:28.048559,Upper Fitness,23,46,50,Oct-07 05:25pm
6,2026-10-07 17:38:28.048629,MAC Court,18,100,18,Oct-07 05:25pm
7,2026-10-07 17:38:28.048691,East Mezzanine (3rd Floor),5,15,34,Oct-07 05:25pm
8,2026-10-07 17:38:28.048756,Atrium Track (3rd Floor),2,50,4,Oct-07 05:25pm
9,2026-10-07 17:38:28.048820,West Mezzanine (3rd Floor),7,25,29,Oct-07 05:25pm


In [26]:
def scrape_facilities():

    url = "https://www.purdue.edu/recwell/facility-usage/index.php"

    driver = webdriver.Chrome()
    driver.get(url)

    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located(
            (By.CLASS_NAME, "rw-c2c-feed__location")
        )
    )

    soup = BeautifulSoup(driver.page_source, "html.parser")

    facility_cards = soup.find_all(
        "div",
        class_="rw-c2c-feed__location"
    )

    facilities = []

    for card in facility_cards:

        name_tag = card.find(
            "h5",
            class_="rw-c2c-feed__location--name"
        )

        capacity_tag = card.find(
            "span",
            class_="rw-c2c-feed__about--capacity"
        )

        updated_tag = card.find(
            "span",
            class_="rw-c2c-feed__about--update"
        )

        if name_tag is None or capacity_tag is None:
            continue

        name = name_tag.get_text(strip=True)
        capacity_text = capacity_tag.get_text(strip=True)

        if updated_tag is not None:
            updated = updated_tag.get_text(strip=True)
        else:
            updated = ""

        match = re.search(
            r"Capacity:\s*(\d+)/(\d+)\s*//\s*(\d+)%",
            capacity_text
        )

        if match:

            facilities.append({
                "timestamp": datetime.now(),
                "facility": name,
                "current": int(match.group(1)),
                "capacity": int(match.group(2)),
                "percent": int(match.group(3)),
                "last_updated": updated.replace(
                    "Last Updated:", ""
                ).strip()
            })

    driver.quit()

    return pd.DataFrame(facilities)

In [28]:
df = scrape_facilities()
df

,timestamp,facility,current,capacity,percent,last_updated
0,2026-10-07 18:02:37.138787,Faculty/Staff Workout Area,5,25,20,Oct-07 05:25pm
1,2026-10-07 18:02:37.138849,Upper Track (4th Floor),1,10,10,Oct-07 05:25pm
2,2026-10-07 18:02:37.138883,Upper Gym,11,100,11,Oct-07 05:25pm
3,2026-10-07 18:02:37.138915,MP6,0,45,0,Oct-07 05:25pm
4,2026-10-07 18:02:37.138945,MP7,0,45,0,Oct-07 05:25pm
5,2026-10-07 18:02:37.138973,Upper Fitness,23,46,50,Oct-07 05:25pm
6,2026-10-07 18:02:37.139001,MAC Court,18,100,18,Oct-07 05:25pm
7,2026-10-07 18:02:37.139030,East Mezzanine (3rd Floor),5,15,34,Oct-07 05:25pm
8,2026-10-07 18:02:37.139059,Atrium Track (3rd Floor),2,50,4,Oct-07 05:25pm
9,2026-10-07 18:02:37.139087,West Mezzanine (3rd Floor),7,25,29,Oct-07 05:25pm


In [29]:
file_name = "crowdcast_data.csv"

df.to_csv(
    file_name,
    mode="a",
    header=not os.path.exists(file_name),
    index=False
)

print("Saved new facility data")

Saved new facility data


In [ ]:
#automatic collection

#every hour
import time
import os
from datetime import datetime

while True:

    df = scrape_facilities()

    df.to_csv(
        "crowdcast_data.csv",
        mode="a",
        header=not os.path.exists("crowdcast_data.csv"),
        index=False
    )

    print("Saved at:", datetime.now())

    #600 seconds = every 10 minutes 
    time.sleep(600)

In [32]:
import pandas as pd

data = pd.read_csv("crowdcast_data.csv")

data.tail(20)

,timestamp,facility,current,capacity,percent,last_updated
308,2026-10-07 19:28:06.702005,B&G Gym 4-6,36,100,36,Oct-07 07:23pm
309,2026-10-07 19:28:06.702034,Scifres,23,80,29,Oct-07 07:23pm
310,2026-10-07 19:28:06.702061,MP2,22,100,22,Oct-07 07:23pm
311,2026-10-07 19:28:06.702088,Climbing Wall,4,18,23,Oct-07 07:23pm
312,2026-10-07 19:28:06.702115,Bouldering Wall,23,26,89,Oct-07 07:23pm
313,2026-10-07 19:28:06.702143,Kilter Board Wall,0,20,0,Oct-07 07:23pm
314,2026-10-07 19:28:06.702171,East Fitness,21,208,11,Oct-07 07:23pm
315,2026-10-07 19:28:06.702199,Olympic Lifting Room,0,16,0,Oct-07 07:23pm
316,2026-10-07 19:28:06.702226,Racquetball Courts 2-3,0,20,0,Oct-07 07:23pm
317,2026-10-07 19:28:06.702253,Climbing Cave,0,15,0,Oct-07 07:23pm


In [33]:
import os
print(os.getcwd())

/Users/nikki/CrowdCast
